# 4 — Neuron List (eps=0.5, cons=0.8)

For AST objects at eps=0.5 and consistency=0.8, list the actual neuron indices in three categories:
- **Concept only** — neurons in the universal circuit but NOT in the token checker
- **Both** — neurons in both universal circuit and token checker
- **Token only** — neurons in the token checker but NOT in the universal circuit

One row per AST object per layer. Saves to CSV.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["h5py", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np
import pandas as pd
import h5py

EPSILON = 0.001
CONSISTENCY = 0.2
LAYERS = [12, 13, 14, 15]  # <-- CHANGE THIS
OBJECT_TYPE = "both"    # <-- "ast", "builtin", or "both"
N_LAYERS = 28

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

OBJ_FILE = f"{PREFIX}3_object_masks_eps{EPSILON}_cons{CONSISTENCY}.h5"
CHK_FILE = f"{PREFIX}3_checker_masks_eps{EPSILON}_cons{CONSISTENCY}.h5"
layers_str = "_".join(str(l) for l in LAYERS)
OUTPUT_FILE = f"{PREFIX}4_neuron_list_eps{EPSILON}_cons{CONSISTENCY}_L{layers_str}_{OBJECT_TYPE}.csv"

os.makedirs(DATA_DIR, exist_ok=True)

print(f"eps={EPSILON}, cons={CONSISTENCY}, layers={LAYERS}, type={OBJECT_TYPE}")
print(f"Output: {OUTPUT_FILE}")

In [ ]:
# Cell 2 – Load universal object masks
def _match_type(ds_name):
    if OBJECT_TYPE == "both":
        return True
    return ds_name.startswith(OBJECT_TYPE)

universal = {}

with h5py.File(os.path.join(DATA_DIR, OBJ_FILE), "r") as f:
    um = f["universal_masks"]
    for lid in LAYERS:
        lk = f"layer_{lid}"
        if lk not in um:
            continue
        for ds_name in um[lk]:
            if not _match_type(ds_name):
                continue
            if ds_name not in universal:
                universal[ds_name] = {}
            universal[ds_name][lid] = np.array(um[lk][ds_name], dtype=bool)

print(f"Loaded {len(universal)} universal circuits ({OBJECT_TYPE}) for layers {LAYERS}")

In [ ]:
# Cell 3 – Load checker masks
checker = {}

with h5py.File(os.path.join(DATA_DIR, CHK_FILE), "r") as f:
    tcm = f["token_checker_masks"]
    for lid in LAYERS:
        lk = f"layer_{lid}"
        if lk not in tcm:
            continue
        for ds_name in tcm[lk]:
            if not _match_type(ds_name):
                continue
            if ds_name not in checker:
                checker[ds_name] = {}
            checker[ds_name][lid] = np.array(tcm[lk][ds_name], dtype=bool)

testable = sorted(set(universal.keys()) & set(checker.keys()))
print(f"Loaded {len(checker)} checker masks ({OBJECT_TYPE})")
print(f"Testable (in both): {len(testable)}")

In [ ]:
# Cell 4 – Build neuron list table (all specified layers)
rows = []

for obj in testable:
    for lid in LAYERS:
        A = universal[obj].get(lid, np.zeros(18944, dtype=bool))
        B = checker[obj].get(lid, np.zeros(18944, dtype=bool))
        
        concept_only = np.where(A & ~B)[0].tolist()
        both = np.where(A & B)[0].tolist()
        token_only = np.where(~A & B)[0].tolist()
        
        rows.append({
            "object": obj,
            "layer": lid,
            "n_concept_only": len(concept_only),
            "n_both": len(both),
            "n_token_only": len(token_only),
            "concept_only": str(concept_only),
            "both": str(both),
            "token_only": str(token_only),
        })

df = pd.DataFrame(rows)
print(f"Layers {LAYERS}: {len(df)} rows ({len(testable)} objects x {len(LAYERS)} layers)")
display(df[["object", "layer", "n_concept_only", "n_both", "n_token_only"]].head(20))

In [ ]:
# Cell 5 – Save CSV
out_path = os.path.join(DATA_DIR, OUTPUT_FILE)
df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")